## Importing data:

In [73]:
import pandas as pd

train_path = '/kaggle/input/competitions/nlp-getting-started/train.csv'
test_path = '/kaggle/input/competitions/nlp-getting-started/test.csv'

train_df = pd.read_csv(train_path)
print('Training set: ',train_df.shape)
display(train_df.head(3))

test_df = pd.read_csv(test_path)
print('Test set: ',test_df.shape)
display(test_df.head(3))

Training set:  (7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1


Test set:  (3263, 4)


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."


In [74]:
print(f"Total samples in the training set are: {len(train_df)}")

Total samples in the training set are: 7613


In [75]:
target = 'target'

#we'll be using bag-of-word model so we will only require text column and the target
X = train_df['text'].values
y = train_df[target].values

## Preparing data:

In [76]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y, 
                                               test_size=0.2,
                                               stratify=y, 
                                               random_state=42)

#converting the arrays from train_test_split to list of stringas, cuz AutoTokenizer doesnt accept numpy arrays or pandas series
X_train = X_train.astype(str).tolist()
X_val = X_val.astype(str).tolist()

We'll tokenize our data using Hugging Face Transformers tokenizer "**AutoTokenizer**"

In [77]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_enc = tokenizer(
    X_train,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

val_enc = tokenizer(
    X_val,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [78]:
#Creating label tensors
import torch

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [79]:
#Creating dataset objects
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    train_enc['input_ids'],
    train_enc['attention_mask'],
    y_train
)

val_dataset = TensorDataset(
    val_enc['input_ids'],
    val_enc['attention_mask'],
    y_val
)

#Loader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

## Model

In [80]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [81]:
#enabling GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Training loop:

In [82]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train() #switching to train mode

for epoch in range(1):
    print(f"\nEpoch {epoch+1}")

    # 🔹 TRAINING
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Train Loss:", loss.item())

    #VALIDATION
    model.eval()  #switching to evaluate mode
    
    total_val_loss = 0

    with torch.no_grad():   
        for batch in val_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print("Val Loss:", avg_val_loss)

    model.train()  # switch back to training mode


Epoch 1
Train Loss: 0.1181013435125351
Val Loss: 0.39733956617678645


## Making predictions on test data:

In [83]:
#Preparing test data
test_enc = tokenizer(
    test_df['text'].astype(str).to_list(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [84]:
test_dataset = TensorDataset(
    test_enc['input_ids'],
    test_enc['attention_mask']
)

test_loader = DataLoader(test_dataset, batch_size=8)

In [85]:
model.eval()

predictions = []

with torch.no_grad():  # no gradients needed
    for batch in test_loader:
        input_ids, attention_mask = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)  # convert to 0/1

        predictions.extend(preds.cpu().numpy())

In [86]:
print(predictions[:10])

[np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]


## Submission

In [87]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: predictions
})
submission.to_csv('submission_transformer_model.csv', index=False)